# E17 — dedicated MT re-translation probe (NLLB-200 + IndicTrans2)

**Bar to beat** on the same rows: Google draft **0.6230** · Codex **0.6483** · **Claude 0.6947**.

Tests one question: *does a purpose-built translation system match the organizers' translator
better than an LLM does?* Claude already beats Google Translate by +0.0717 — so the target is
**not** Google output. Whose fingerprint is it?

### Fixes in v3 — the previous two runs both died
| Run | Failure | Fix |
|---|---|---|
| v1 | `401` — IndicTrans2 is a **gated** HF repo | token supplied; NLLB (ungated) runs first regardless |
| v2 | **CUDA OOM** — `batch 24 × beams 5 × max_len 512` kept 120 sequences alive with a 512-token KV cache, plus 3.42 GB lost to fragmentation | `batch 8 × beams 4 × max_len 200`, `expandable_segments`, length-sorted batches, and an automatic batch-halving retry |

A single sentence needs ~80 Bengali tokens — `max_length 512` was 6× oversized, which is where
the memory went.

### Precision
**fp32.** T4 is sm_75 — no bf16 hardware — and this project already lost a submission cycle to a
*silent* fp16 NaN that still wrote a well-formed CSV. 1.3B is 5.2 GB fp32, leaving ~9 GB.

In [ ]:
# 1 ── allocator config MUST be set before torch is imported, then the hardware gate
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # v2 lost 3.42 GB to fragmentation

import torch, gc
assert torch.cuda.is_available(), "no GPU — enable the accelerator"
cap = torch.cuda.get_device_capability()
TOTAL = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {torch.cuda.get_device_name(0)} | sm_{cap[0]}{cap[1]} | {TOTAL:.2f} GB | torch {torch.__version__}")
assert cap[0] >= 7, f"sm_{cap[0]}{cap[1]} unsupported — Kaggle's torch ships no sm_60 kernels (P100)"

def vram(tag=""):
    print(f"   [vram] {tag} alloc {torch.cuda.memory_allocated()/1e9:.2f} GB "
          f"| reserved {torch.cuda.memory_reserved()/1e9:.2f} GB / {TOTAL:.2f} GB")

def free_gpu(*objs):
    for o in objs:
        del o
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

In [ ]:
# 2 ── deps + credentials
!pip install -q "transformers==4.57.3" sentencepiece sacremoses
import transformers
assert transformers.__version__ == "4.57.3", transformers.__version__
print("transformers", transformers.__version__)

# Prefer a Kaggle Secret (Add-ons -> Secrets -> HF_TOKEN). Falls back to the inline
# token so the run is not blocked on a manual UI step.
# ⚠️ This token is visible in the notebook source — rotate it once E17 is settled.
HF_TOKEN = "<REDACTED: use a Kaggle Secret named HF_TOKEN>"
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("using HF_TOKEN from Kaggle Secrets")
except Exception:
    print("using inline HF token (no Kaggle Secret set)")

In [ ]:
# 3 ── data + sentence split (both models are sentence-level; a 108-word paragraph
#      degrades them badly)
import glob, re, time, pandas as pd

N_ROWS = 200                      # 1000 for the larger probe
pat = f"/kaggle/input/**/PROBE_{'1000' if N_ROWS > 200 else '200'}_dev.csv"
src = glob.glob(pat, recursive=True)
assert src, f"no match for {pat} — attach the nascenia-e17-probe dataset"
rows = pd.read_csv(src[0]).head(N_ROWS)

SENT = re.compile(r"(?<=[.!?])\s+")
def sentences(t):
    p = [s.strip() for s in SENT.split(str(t).strip()) if s.strip()]
    return p or [str(t).strip() or "."]

flat, owner = [], []
for i, en in enumerate(rows["english"]):
    for s in sentences(en):
        flat.append(s); owner.append(i)

# Length-sorted execution order: batching similar lengths together cuts padding
# waste sharply. Results are mapped back to original order before rejoining.
order = sorted(range(len(flat)), key=lambda i: len(flat[i]))
w = pd.Series(flat).str.split().str.len()
print(f"{src[0]}\n  {len(rows)} rows -> {len(flat)} sentences ({len(flat)/len(rows):.1f}/row)")
print(f"  sentence words: mean {w.mean():.0f} p95 {w.quantile(.95):.0f} max {w.max()}")

In [ ]:
# 4 ── shared runner, with OOM survival
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

BATCH, BEAMS, MAX_LEN = 8, 4, 200   # v2 used 24 / 5 / 512 and OOM'd

def run_batches(fn, label):
    """Translate in length-sorted batches; on OOM, halve the batch and retry that
    chunk rather than losing the whole run."""
    got, t0, i = {}, time.time(), 0
    while i < len(order):
        bs, idx = BATCH, order[i:i+BATCH]
        while True:
            try:
                out = fn([flat[j] for j in idx])
                break
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache(); bs //= 2
                assert bs >= 1, "OOM even at batch 1 — use nllb-200-distilled-600M"
                idx = idx[:bs]
                print(f"    ⚠️ OOM -> retrying at batch {bs}", flush=True)
        got.update(dict(zip(idx, out)))
        i += len(idx)
        if (i // BATCH) % 10 == 0 or i >= len(order):
            el = time.time() - t0
            print(f"  {label} {i}/{len(order)}  {el/60:.1f} min "
                  f"(eta {el/max(i,1)*(len(order)-i)/60:.1f} min)", flush=True)
    print(f"  {label} done in {(time.time()-t0)/60:.1f} min"); vram(label)
    return [got[i] for i in range(len(flat))]      # back to original order

def rejoin_and_write(pieces, tag):
    assert len(pieces) == len(flat), f"{len(pieces)} pieces != {len(flat)} sentences"
    joined = [[] for _ in range(len(rows))]
    for o, p in zip(owner, pieces):
        joined[o].append(p)
    bn = [" ".join(x) for x in joined]

    # 🔴 id alignment is the whole approach — one dropped row breaks it silently
    assert len(bn) == len(rows), f"❌ {len(bn)} outputs for {len(rows)} rows"
    assert all(str(x).strip() for x in bn), "❌ blank translation"

    out = pd.DataFrame({"hcm_id": rows["hcm_id"].values, "bengali": bn})
    out.to_csv(f"{tag}_TRANSLATED.csv", index=False, encoding="utf-8")
    s = out.bengali.astype(str)
    print(f"\n✅ {tag}_TRANSLATED.csv | {len(out)} rows | mean {s.str.split().str.len().mean():.0f} words")
    print(f"   🔴 contamination — হেলো {s.str.startswith('হেলো').mean()*100:.1f}% (target 76.4%) · "
          f"নাসেনিয়া {s.str.contains('নাসেনিয়া').mean()*100:.1f}% (target 50.0%) — both must be 0.0%")
    print(f"   sample: {out.bengali[0][:180]}")

print(f"batch {BATCH} · beams {BEAMS} · max_len {MAX_LEN}")

In [ ]:
# 5 ── ARM A: NLLB-200 (ungated). Runs first so its output is on disk before the
#      gated arm is even attempted.
NLLB = "facebook/nllb-200-distilled-1.3B"
tok = AutoTokenizer.from_pretrained(NLLB, src_lang="eng_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained(NLLB, torch_dtype=torch.float32).to("cuda").eval()

# lang_code_to_id was removed in newer transformers — resolve defensively. A wrong
# id yields fluent output in the WRONG LANGUAGE, which scores as a bad translation.
bos = tok.convert_tokens_to_ids("ben_Beng")
if bos is None or bos == tok.unk_token_id:
    bos = tok.lang_code_to_id["ben_Beng"]
print(f"{NLLB} | {sum(p.numel() for p in model.parameters())/1e9:.2f}B | "
      f"{next(model.parameters()).dtype} | ben_Beng={bos}"); vram("nllb loaded")

def nllb_batch(chunk):
    enc = tok(chunk, truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        gen = model.generate(**enc, forced_bos_token_id=bos, num_beams=BEAMS, max_length=MAX_LEN)
    return tok.batch_decode(gen, skip_special_tokens=True)

print("smoke:", nllb_batch(["The patient has a fever and a persistent cough."])[0])
rejoin_and_write(run_batches(nllb_batch, "nllb"), "nllb13b")
free_gpu(model, tok); vram("after free")

In [ ]:
# 6 ── ARM B: IndicTrans2 (gated). Wrapped so a failure here can never lose Arm A.
try:
    !pip install -q git+https://github.com/VarunGumma/IndicTransToolkit.git
    from IndicTransToolkit.processor import IndicProcessor
    IT2 = "ai4bharat/indictrans2-en-indic-1B"
    t2 = AutoTokenizer.from_pretrained(IT2, trust_remote_code=True, token=HF_TOKEN)
    m2 = AutoModelForSeq2SeqLM.from_pretrained(
        IT2, trust_remote_code=True, torch_dtype=torch.float32, token=HF_TOKEN
    ).to("cuda").eval()
    ip = IndicProcessor(inference=True)
    print(f"{IT2} | {sum(p.numel() for p in m2.parameters())/1e9:.2f}B"); vram("it2 loaded")

    def it2_batch(chunk):
        pre = ip.preprocess_batch(chunk, src_lang="eng_Latn", tgt_lang="ben_Beng")
        enc = t2(pre, truncation=True, max_length=MAX_LEN, padding="longest", return_tensors="pt").to("cuda")
        with torch.no_grad():
            gen = m2.generate(**enc, num_beams=BEAMS, max_length=MAX_LEN, num_return_sequences=1)
        return ip.postprocess_batch(t2.batch_decode(gen, skip_special_tokens=True), lang="ben_Beng")

    print("smoke:", it2_batch(["The patient has a fever and a persistent cough."])[0])
    rejoin_and_write(run_batches(it2_batch, "it2"), "indictrans2")
except Exception as e:
    print(f"⚠️ IndicTrans2 arm failed — {type(e).__name__}: {str(e)[:300]}")
    print("   Arm A (NLLB) output above is unaffected — download it and score.")

In [ ]:
# 7 ── what to download
print("Download into E17_RETRANSLATE/, then run:  python score_all.py\n")
found = sorted(glob.glob("*_TRANSLATED.csv"))
for f in found:
    print(f"  {f}  ({os.path.getsize(f)/1024:.0f} KB, {len(pd.read_csv(f))} rows)")
assert found, "❌ nothing produced — both arms failed"